# Feature Engineering

## Objective

Feature engineering is the process of creating new variables from existing features to better represent the underlying patterns in the data. These engineered features can improve the predictive performance of machine learning models by capturing important property characteristics.

In [41]:
import pandas as pd

In [42]:
df = pd.read_csv("../data/cleaned_housing_data.csv")

df.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Total Bathrooms,Total Living Area
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,NaN,NaN,0,5,2010,WD,Normal,215000,2.0,2736.0
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,MnPrv,NaN,0,6,2010,WD,Normal,105000,1.0,1778.0
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,NaN,Gar2,12500,6,2010,WD,Normal,172000,1.5,2658.0
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,NaN,NaN,0,4,2010,WD,Normal,244000,3.5,4220.0
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,MnPrv,NaN,0,3,2010,WD,Normal,189900,2.5,2557.0


In [43]:
df["House Age"] = df["Yr Sold"] - df["Year Built"]

df[["Year Built", "Yr Sold", "House Age"]].head()

,Year Built,Yr Sold,House Age
0,1960,2010,50
1,1961,2010,49
2,1958,2010,52
3,1968,2010,42
4,1997,2010,13


In [44]:
df["Years Since Remodel"] = df["Yr Sold"] - df["Year Remod/Add"]

df[["Year Remod/Add", "Yr Sold", "Years Since Remodel"]].head()

,Year Remod/Add,Yr Sold,Years Since Remodel
0,1960,2010,50
1,1961,2010,49
2,1958,2010,52
3,1968,2010,42
4,1998,2010,12


In [45]:
df["House Remodeled"] = (
    df["Year Built"] != df["Year Remod/Add"]
).astype(int)

df["House Remodeled"].value_counts()

House Remodeled
0    1569
1    1361
Name: count, dtype: int64

In [46]:
df["Total Bathrooms"] = (
    df["Full Bath"]
    + (0.5 * df["Half Bath"])
    + df["Bsmt Full Bath"]
    + (0.5 * df["Bsmt Half Bath"])
)

df["Total Bathrooms"].describe()

count    2930.000000
mean        2.217918
std         0.807444
min         1.000000
25%         1.500000
50%         2.000000
75%         2.500000
max         7.000000
Name: Total Bathrooms, dtype: float64

In [47]:
df["Total Porch Area"] = (
    df["Open Porch SF"]
    + df["Enclosed Porch"]
    + df["3Ssn Porch"]
    + df["Screen Porch"]
)

df["Total Porch Area"].describe()

count    2930.000000
mean       89.139590
std       107.734138
min         0.000000
25%         0.000000
50%        50.000000
75%       136.000000
max      1207.000000
Name: Total Porch Area, dtype: float64

In [48]:
df["Total Living Area"] = (
    df["Gr Liv Area"]
    + df["Total Bsmt SF"]
)

df["Total Living Area"].describe()

count     2930.000000
mean      2550.946075
std        805.253248
min        334.000000
25%       2004.000000
50%       2452.000000
75%       2994.500000
max      11752.000000
Name: Total Living Area, dtype: float64

In [49]:
new_features = [
    "House Age",
    "Years Since Remodel",
    "House Remodeled",
    "Total Bathrooms",
    "Total Porch Area",
    "Total Living Area"
]

df[new_features].head()

,House Age,Years Since Remodel,House Remodeled,Total Bathrooms,Total Porch Area,Total Living Area
0,50,50,0,2.0,62,2736.0
1,49,49,0,1.0,120,1778.0
2,52,52,0,1.5,36,2658.0
3,42,42,0,3.5,0,4220.0
4,13,12,1,2.5,34,2557.0


# Feature Engineering – Summary

## New Features Created

- **House Age**: Number of years between construction and sale.
- **Years Since Remodel**: Number of years since the last remodeling.
- **House Remodeled**: Binary feature indicating whether the house has been remodeled.
- **Total Bathrooms**: Combined full and half bathrooms, including basement bathrooms.
- **Total Porch Area**: Sum of all porch-related areas.
- **Total Living Area**: Combined above-ground and basement living area.

## Outcome

These engineered features provide additional information about the property's age, renovation history, living space, and amenities. They are expected to improve the model's ability to predict house prices.

# Encode Categorical Variables

## Objective

Machine learning models require numerical input. This section identifies categorical features and converts them into numerical representations using One-Hot Encoding. This transformation enables the model to learn from categorical information without introducing artificial ordering.

In [50]:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Number of categorical features:", len(categorical_cols))
print(categorical_cols.tolist())

Number of categorical features: 43
['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC', 'Central Air', 'Electrical', 'Kitchen Qual', 'Functional', 'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature', 'Sale Type', 'Sale Condition']


In [51]:
print("Dataset Shape Before Encoding:", df.shape)

Dataset Shape Before Encoding: (2930, 86)


In [52]:
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [53]:
print("Dataset Shape After Encoding:", df.shape)

Dataset Shape After Encoding: (2930, 267)


In [54]:
print(df.dtypes.value_counts())

int64      254
float64     13
Name: count, dtype: int64


In [55]:
remaining_cat = df.select_dtypes(include=["object"]).columns

print("Remaining categorical columns:", len(remaining_cat))
print(remaining_cat.tolist())

Remaining categorical columns: 0
[]


# Categorical Encoding – Summary

## Encoding Method

All categorical features were transformed using **One-Hot Encoding** with `pd.get_dummies()`.

## Why One-Hot Encoding?

- Converts categorical values into numerical format.
- Prevents the model from assuming an order between categories.
- Makes the dataset compatible with machine learning algorithms.

## Outcome

- All categorical variables were successfully encoded.
- The dataset now contains only numerical features.
- The processed dataset is ready for feature scaling and model training.

# Feature Scaling

## Objective

Feature scaling standardizes numerical features so that they have a similar range. This prevents features with larger values from dominating those with smaller values and improves the performance of many machine learning algorithms.

In [56]:
X = df.drop("SalePrice", axis=1)

y = df["SalePrice"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (2930, 266)
Target Shape: (2930,)


In [57]:
from sklearn.preprocessing import StandardScaler

In [58]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [59]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Sale Type_ConLw,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,-0.877005,3.375742,2.744381,-0.067254,-0.506718,-0.375537,-1.163488,0.061046,0.431433,-0.293918,...,-0.052324,-0.298018,-0.048937,-0.018477,0.394161,-0.064128,-0.090878,-0.126294,0.462878,-0.302072
1,-0.877005,0.514952,0.187097,-0.776079,0.393091,-0.342468,-1.115542,-0.566039,0.056029,0.557582,...,-0.052324,-0.298018,-0.048937,-0.018477,0.394161,-0.064128,-0.090878,-0.126294,0.462878,-0.302072
2,-0.877005,0.561850,0.522814,-0.067254,0.393091,-0.441674,-1.259380,0.038650,1.054912,-0.293918,...,-0.052324,-0.298018,-0.048937,-0.018477,0.394161,-0.064128,-0.090878,-0.126294,0.462878,-0.302072
3,-0.877005,1.124628,0.128458,0.641571,-0.506718,-0.110988,-0.779919,-0.566039,1.366651,-0.293918,...,-0.052324,-0.298018,-0.048937,-0.018477,0.394161,-0.064128,-0.090878,-0.126294,0.462878,-0.302072
4,0.061285,0.233563,0.467348,-0.776079,-0.506718,0.848000,0.658466,-0.566039,0.765126,-0.293918,...,-0.052324,-0.298018,-0.048937,-0.018477,0.394161,-0.064128,-0.090878,-0.126294,0.462878,-0.302072


In [60]:
print("Mean (first 5 columns):")
print(X_scaled.iloc[:, :5].mean())

print("\nStandard Deviation (first 5 columns):")
print(X_scaled.iloc[:, :5].std())

Mean (first 5 columns):
MS SubClass    -9.700242e-18
Lot Frontage    2.825196e-16
Lot Area        5.820145e-17
Overall Qual   -8.002700e-17
Overall Cond    2.691817e-16
dtype: float64

Standard Deviation (first 5 columns):
MS SubClass     1.000171
Lot Frontage    1.000171
Lot Area        1.000171
Overall Qual    1.000171
Overall Cond    1.000171
dtype: float64


In [61]:
print("Scaled Dataset Shape:", X_scaled.shape)

Scaled Dataset Shape: (2930, 266)


In [62]:
processed_data = pd.concat([X_scaled, y], axis=1)

processed_data.to_csv(
    "../data/processed_housing_data.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


# Feature Scaling – Summary

## Scaling Method

The numerical features were standardized using **StandardScaler** from Scikit-learn.

## Why StandardScaler?

- Centers each feature around a mean of 0.
- Scales each feature to have a standard deviation of 1.
- Improves the performance of many machine learning algorithms.

## Outcome

- All predictor variables have been standardized.
- The target variable (`SalePrice`) was left unchanged.
- The processed dataset has been saved and is ready for model training.